## Modelo Relacional Mejorado - Sistema de Ventas TechCore

### FASE 1: Configuración Inicial


In [19]:
# Importación de librerías
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import warnings
import os
import re

# Configuración de advertencias
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [ ]:
# Instalar dependencia si es necesario
try:
    import openpyxl
    print("✅ openpyxl ya está instalado")
except ImportError:
    print("📦 Instalando openpyxl...")
    !pip install openpyxl
    import openpyxl
    print("✅ openpyxl instalado correctamente")

### FASE 2: Carga y Análisis del Dataset

In [20]:
# Cargar el dataset transformado
try:
    df_ventas = pd.read_csv('ventasTransformed.csv', encoding='utf-8')
    print("✅ Dataset cargado correctamente")
except:
    # Intentar con otra codificación si falla
    df_ventas = pd.read_csv('ventasTransformed.csv', encoding='latin-1')
    print("✅ Dataset cargado con codificación latin-1")

print("📊 ANÁLISIS INICIAL DEL DATASET")
print("=" * 50)
print(f"• Dimensiones: {df_ventas.shape[0]} filas, {df_ventas.shape[1]} columnas")
print(f"• Rango de años: {df_ventas['Año'].min()} - {df_ventas['Año'].max()}")
print(f"• Productos únicos: {df_ventas['Producto1Nombre'].nunique()}")
print(f"• Clientes únicos: {df_ventas['ClienteNombre'].nunique()}")
print(f"• Sucursales únicas: {df_ventas['SucursalNombre'].nunique()}")
print(f"• Vendedores únicos: {df_ventas['VendedorNombre'].nunique()}")

# Mostrar estructura del dataset
print("\n🔍 MUESTRA INICIAL DEL DATASET:")
print(df_ventas.head(3))

✅ Dataset cargado correctamente
📊 ANÁLISIS INICIAL DEL DATASET
• Dimensiones: 30000 filas, 37 columnas
• Rango de años: 2014 - 2025
• Productos únicos: 40
• Clientes únicos: 17449
• Sucursales únicas: 6
• Vendedores únicos: 30

🔍 MUESTRA INICIAL DEL DATASET:
   Producto1Cantidad  Producto2Cantidad  Producto3Cantidad CiudadSucursal  \
0                  1                  0                  0         Bogota   
1                  1                  0                  0         Bogota   
2                  1                  0                  0         Bogota   

         ClienteNombre  DescuentoVenta DireccionCliente  EdadCliente  \
0    Aarón del Márquez               0    cll 52 #49-31           32   
1  Aarón Herrera Tello               1     cll 2 #72-64           37   
2   Aarón Luna Nicolau               0    cll 79 #71-25           29   

          EmailCliente   Año  ... Producto3Subtotal      SucursalNombre  \
0                  NaN  2016  ...                 0  TechCore Bogota

### FASE 3: Diseño del Modelo Entidad-Relación

#### DIAGRAMA ENTIDAD-RELACIÓN (Modelo Estrella)

### FASE 4: Creación de Tablas Dimensión
#### 4.1 Tabla Ciudades

In [21]:
# Crear tabla de ciudades
df_ciudades = df_ventas[['CiudadSucursal']].drop_duplicates().reset_index(drop=True)
df_ciudades['CiudadID'] = range(1, len(df_ciudades) + 1)
df_ciudades = df_ciudades[['CiudadID', 'CiudadSucursal']]
df_ciudades.columns = ['CiudadID', 'NombreCiudad']

print("🏙️ TABLA CIUDADES")
print("=" * 30)
print(f"• {len(df_ciudades)} ciudades únicas")
print("\nCiudades identificadas:")
print(df_ciudades)

🏙️ TABLA CIUDADES
• 5 ciudades únicas

Ciudades identificadas:
   CiudadID NombreCiudad
0         1       Bogota
1         2         Cali
2         3     Medellin
3         4          NaN
4         5      Pereira


### 4.2 Función de Asignación Inteligente de Ciudades

In [22]:
def asignar_ciudad_inteligente(sucursal_nombre, ciudades_disponibles):
    """
    Asigna la ciudad de forma inteligente:
    1. Si la sucursal contiene el nombre de una ciudad conocida, usar esa ciudad
    2. Si no, usar la ciudad más frecuente
    """
    # Primero verificar si hay ciudades disponibles
    if len(ciudades_disponibles) == 0:
        return 'No especificado'
    
    ciudades_conocidas = ['Medellín', 'Bogotá', 'Pereira', 'Cali', 'Bogota', 'Medellin']
    
    for ciudad in ciudades_conocidas:
        if ciudad.lower() in sucursal_nombre.lower():
            # Mapear variaciones a nombres consistentes
            if ciudad in ['Bogota', 'Bogotá']:
                return 'Bogotá'
            elif ciudad in ['Medellin', 'Medellín']:
                return 'Medellín'
            return ciudad
    
    # Si no contiene ciudad conocida, usar la más frecuente
    return ciudades_disponibles.value_counts().index[0]

print("✅ Función de asignación inteligente creada")

✅ Función de asignación inteligente creada


### 4.3 Tabla Sucursales

In [23]:
print("🏪 CREANDO TABLA SUCURSALES MEJORADA")
print("=" * 50)

def normalizar_nombre_sucursal(sucursal_nombre):
    """
    Normaliza y mejora los nombres de sucursales para mejor identificación
    """
    # Manejar valores nulos
    if pd.isna(sucursal_nombre):
        return 'Sucursal No Especificada'
    
    sucursal_lower = str(sucursal_nombre).lower()
    
    # Mapeo de nombres mejorado
    if 'medellín' in sucursal_lower or 'medellin' in sucursal_lower:
        if '#1' in sucursal_lower or ' 1' in sucursal_lower:
            return 'TechCore Medellín #1'
        elif '#2' in sucursal_lower or ' 2' in sucursal_lower:
            return 'TechCore Medellín #2'
        else:
            # Si no tiene número, asignar #1 por defecto
            return 'TechCore Medellín #1'
    
    elif 'bogotá' in sucursal_lower or 'bogota' in sucursal_lower:
        if '#1' in sucursal_lower or ' 1' in sucursal_lower:
            return 'TechCore Bogotá #1'
        elif '#2' in sucursal_lower or ' 2' in sucursal_lower:
            return 'TechCore Bogotá #2'
        else:
            # Si no tiene número, asignar #1 por defecto
            return 'TechCore Bogotá #1'
    
    elif 'cali' in sucursal_lower:
        return 'TechCore Cali'
    
    elif 'pereira' in sucursal_lower:
        return 'TechCore Pereira'
    
    else:
        return str(sucursal_nombre)  # Mantener original si no coincide

def asignar_ciudad_mejorada(sucursal_nombre):
    """
    Asigna ciudad basada en el nombre de sucursal normalizado
    """
    sucursal_normalizado = normalizar_nombre_sucursal(sucursal_nombre)
    sucursal_lower = str(sucursal_normalizado).lower()
    
    if 'medellín' in sucursal_lower:
        return 'Medellín'
    elif 'bogotá' in sucursal_lower:
        return 'Bogotá'
    elif 'cali' in sucursal_lower:
        return 'Cali'
    elif 'pereira' in sucursal_lower:
        return 'Pereira'
    else:
        return 'No especificado'

# Verificar y limpiar datos de sucursales en el dataset original
print("🔍 Verificando datos de sucursales originales...")
print(f"Valores nulos en SucursalNombre: {df_ventas['SucursalNombre'].isnull().sum()}")
print(f"Valores únicos en SucursalNombre: {df_ventas['SucursalNombre'].nunique()}")

# Aplicar normalización a los nombres de sucursales
print("🔄 Normalizando nombres de sucursales...")
sucursales_mejoradas = []

# Primero, obtener todas las sucursales únicas y limpiarlas
sucursales_unicas = df_ventas['SucursalNombre'].unique()
print(f"Sucursales únicas encontradas: {len(sucursales_unicas)}")

for sucursal in sucursales_unicas:
    nombre_normalizado = normalizar_nombre_sucursal(sucursal)
    ciudad_asignada = asignar_ciudad_mejorada(nombre_normalizado)
    
    sucursales_mejoradas.append({
        'SucursalOriginal': sucursal,
        'SucursalNombre': nombre_normalizado,
        'CiudadSucursal': ciudad_asignada
    })

# Crear DataFrame de sucursales mejorado
df_sucursales_mejorado = pd.DataFrame(sucursales_mejoradas).drop_duplicates(subset=['SucursalNombre'])
df_sucursales_mejorado['SucursalID'] = range(1, len(df_sucursales_mejorado) + 1)

print(f"\n✅ Sucursales normalizadas:")
for _, row in df_sucursales_mejorado.iterrows():
    print(f"  • '{row['SucursalOriginal']}' -> '{row['SucursalNombre']}' -> {row['CiudadSucursal']}")

# Mapear CiudadID con manejo robusto de errores
ciudad_map = dict(zip(df_ciudades['NombreCiudad'], df_ciudades['CiudadID']))
print(f"\n📋 Ciudades disponibles en el mapeo: {list(ciudad_map.keys())}")

def mapear_ciudad_seguro(ciudad_nombre):
    """Mapea ciudad de forma segura, manejando todos los casos edge"""
    # Manejar valores nulos
    if pd.isna(ciudad_nombre):
        print(f"⚠️  Ciudad nula encontrada, asignando valor por defecto")
        return 1
    
    ciudad_str = str(ciudad_nombre).strip()
    
    # Buscar coincidencia exacta primero
    if ciudad_str in ciudad_map:
        return ciudad_map[ciudad_str]
    
    # Buscar coincidencia parcial (case insensitive)
    for ciudad_existente in ciudad_map.keys():
        if ciudad_existente and not pd.isna(ciudad_existente):
            try:
                if (ciudad_str.lower() in str(ciudad_existente).lower() or 
                    str(ciudad_existente).lower() in ciudad_str.lower()):
                    print(f"🔍 Coincidencia parcial: '{ciudad_str}' -> '{ciudad_existente}'")
                    return ciudad_map[ciudad_existente]
            except AttributeError as e:
                print(f"❌ Error procesando ciudad: '{ciudad_str}', existente: '{ciudad_existente}'. Error: {e}")
                continue
    
    print(f"⚠️  Ciudad no encontrada: '{ciudad_str}', asignando valor por defecto")
    return 1  # Valor por defecto

print("\n🔄 Mapeando ciudades a IDs...")
df_sucursales_mejorado['CiudadID'] = df_sucursales_mejorado['CiudadSucursal'].apply(mapear_ciudad_seguro)

# Seleccionar columnas finales
df_sucursales_mejorado = df_sucursales_mejorado[['SucursalID', 'SucursalNombre', 'CiudadID']]

# Reemplazar la tabla original
df_sucursales = df_sucursales_mejorado

print(f"\n✅ TABLA SUCURSALES MEJORADA CREADA")
print(f"• {len(df_sucursales)} sucursales únicas")

# Verificar las asignaciones finales
print("\n🏪 SUCURSALES FINALES IDENTIFICADAS:")
for _, row in df_sucursales.iterrows():
    ciudad_info = df_ciudades[df_ciudades['CiudadID'] == row['CiudadID']]
    if len(ciudad_info) > 0:
        ciudad_nombre = ciudad_info['NombreCiudad'].values[0]
        print(f"  • {row['SucursalID']:2d}. {row['SucursalNombre']:25} -> {ciudad_nombre}")
    else:
        print(f"  ⚠️  {row['SucursalID']:2d}. {row['SucursalNombre']:25} -> CiudadID {row['CiudadID']} (no encontrada)")

# Actualizar el dataset original con los nombres normalizados
print("\n🔄 Actualizando dataset original con nombres normalizados...")
sucursal_rename_map = dict(zip(sucursales_unicas, 
                              [normalizar_nombre_sucursal(s) for s in sucursales_unicas]))
df_ventas['SucursalNombre'] = df_ventas['SucursalNombre'].map(sucursal_rename_map)

# Verificar que no hay valores nulos después de la normalización
print(f"✅ Valores nulos en SucursalNombre después de normalización: {df_ventas['SucursalNombre'].isnull().sum()}")

# Mostrar distribución final de sucursales
print(f"\n📊 DISTRIBUCIÓN FINAL DE SUCURSALES:")
sucursales_counts = df_ventas['SucursalNombre'].value_counts()
for sucursal, count in sucursales_counts.items():
    print(f"  • {sucursal:25} : {count:>4} ventas")

# %% [markdown]
# ### 4.4 Verificación de la Normalización de Sucursales

# %%
print("🔍 VERIFICACIÓN DE LA NORMALIZACIÓN DE SUCURSALES")
print("=" * 55)

# Agrupar sucursales por ciudad para verificar la asignación
print("\n🏙️  SUCURSALES AGRUPADAS POR CIUDAD:")
for ciudad_id in df_sucursales['CiudadID'].unique():
    sucursales_ciudad = df_sucursales[df_sucursales['CiudadID'] == ciudad_id]
    ciudad_nombre = df_ciudades[df_ciudades['CiudadID'] == ciudad_id]['NombreCiudad'].values[0]
    
    print(f"\n📍 {ciudad_nombre}:")
    for _, sucursal in sucursales_ciudad.iterrows():
        # Contar cuántas ventas tiene cada sucursal
        count_ventas = len(df_ventas[df_ventas['SucursalNombre'] == sucursal['SucursalNombre']])
        print(f"   • {sucursal['SucursalNombre']:25} ({count_ventas:>4} ventas)")

# Verificar específicamente Medellín y Bogotá
print(f"\n🔍 COMPROBACIÓN ESPECÍFICA:")
sucursales_medellin = [s for s in df_sucursales['SucursalNombre'] if 'Medellín' in s]
sucursales_bogota = [s for s in df_sucursales['SucursalNombre'] if 'Bogotá' in s]

print(f"✅ Sucursales de Medellín diferenciadas: {sucursales_medellin}")
print(f"✅ Sucursales de Bogotá diferenciadas: {sucursales_bogota}")

# Verificar que todas las sucursales tengan una ciudad asignada
print(f"\n✅ VERIFICACIÓN FINAL:")
print(f"• Total sucursales: {len(df_sucursales)}")
print(f"• Sucursales sin CiudadID: {(df_sucursales['CiudadID'] == 1).sum()}")
print(f"• Sucursales con CiudadID válido: {(df_sucursales['CiudadID'] != 1).sum()}")

🏪 CREANDO TABLA SUCURSALES MEJORADA
🔍 Verificando datos de sucursales originales...
Valores nulos en SucursalNombre: 0
Valores únicos en SucursalNombre: 6
🔄 Normalizando nombres de sucursales...
Sucursales únicas encontradas: 6

✅ Sucursales normalizadas:
  • 'TechCore Bogota #1' -> 'TechCore Bogotá #1' -> Bogotá
  • 'TechCore Bogota #2' -> 'TechCore Bogotá #2' -> Bogotá
  • 'TechCore Cali' -> 'TechCore Cali' -> Cali
  • 'TechCore Medellin #2' -> 'TechCore Medellín #2' -> Medellín
  • 'TechCore Medellin #1' -> 'TechCore Medellín #1' -> Medellín
  • 'TechCore Pereira' -> 'TechCore Pereira' -> Pereira

📋 Ciudades disponibles en el mapeo: ['Bogota', 'Cali', 'Medellin', nan, 'Pereira']

🔄 Mapeando ciudades a IDs...
⚠️  Ciudad no encontrada: 'Bogotá', asignando valor por defecto
⚠️  Ciudad no encontrada: 'Bogotá', asignando valor por defecto
⚠️  Ciudad no encontrada: 'Medellín', asignando valor por defecto
⚠️  Ciudad no encontrada: 'Medellín', asignando valor por defecto

✅ TABLA SUCURSALES

### 4.4 Tabla Clientes

In [24]:
# Crear tabla de clientes
df_clientes = df_ventas[['ClienteNombre', 'GeneroCliente', 'EdadCliente', 'EmailCliente', 'TelefonoCliente', 'DireccionCliente']].copy()
df_clientes = df_clientes.drop_duplicates(subset=['ClienteNombre']).reset_index(drop=True)

# Limpiar y estandarizar datos de clientes
df_clientes['EmailCliente'] = df_clientes['EmailCliente'].fillna('No especificado')
df_clientes['TelefonoCliente'] = df_clientes['TelefonoCliente'].fillna('No especificado')
df_clientes['DireccionCliente'] = df_clientes['DireccionCliente'].fillna('No especificado')

# Crear columna RangoEdad
def asignar_rango_edad(edad):
    if pd.isna(edad):
        return 'No especificado'
    edad = int(edad)
    if 18 <= edad <= 24:
        return '18-24'
    elif 25 <= edad <= 34:
        return '25-34'
    elif 35 <= edad <= 44:
        return '35-44'
    elif 45 <= edad <= 51:
        return '45-51'
    else:
        return 'Fuera de rango'

df_clientes['RangoEdad'] = df_clientes['EdadCliente'].apply(asignar_rango_edad)

df_clientes['ClienteID'] = range(1, len(df_clientes) + 1)
df_clientes = df_clientes[['ClienteID', 'ClienteNombre', 'GeneroCliente', 'EdadCliente', 'RangoEdad', 'TelefonoCliente', 'EmailCliente', 'DireccionCliente']]
df_clientes.columns = ['ClienteID', 'Nombre', 'Genero', 'Edad', 'RangoEdad', 'Telefono', 'Email', 'Direccion']

print("👥 TABLA CLIENTES")
print("=" * 30)
print(f"• {len(df_clientes)} clientes únicos")
print(f"• Distribución por rango de edad:")
print(df_clientes['RangoEdad'].value_counts())
print("\nMuestra de clientes:")
print(df_clientes.head(5))

👥 TABLA CLIENTES
• 17449 clientes únicos
• Distribución por rango de edad:
RangoEdad
35-44    5146
25-34    5049
18-24    3680
45-51    3574
Name: count, dtype: int64

Muestra de clientes:
   ClienteID                Nombre Genero  Edad RangoEdad           Telefono  \
0          1     Aarón del Márquez      M    32     25-34  '+34 806 97 55 19   
1          2   Aarón Herrera Tello      M    37     35-44   '+34 826 474 028   
2          3    Aarón Luna Nicolau      M    29     25-34   '+34925 79 10 06   
3          4  Aarón Sevilla García      M    28     25-34   '+34 876 688 797   
4          5           Abel Blanes      M    48     45-51      '+34942482140   

                 Email      Direccion  
0      No especificado  cll 52 #49-31  
1  aarón11@hotmail.com   cll 2 #72-64  
2    aarón82@yahoo.com  cll 79 #71-25  
3    aarón78@yahoo.com  cll 92 #62-21  
4      No especificado  cra 68 #62-54  


### 4.5 Tabla Vendedores

In [25]:
# Crear tabla de vendedores
df_vendedores = df_ventas[['VendedorNombre']].drop_duplicates().reset_index(drop=True)
df_vendedores['VendedorID'] = range(1, len(df_vendedores) + 1)
df_vendedores = df_vendedores[['VendedorID', 'VendedorNombre']]
df_vendedores.columns = ['VendedorID', 'NombreVendedor']

print("👨‍💼 TABLA VENDEDORES")
print("=" * 30)
print(f"• {len(df_vendedores)} vendedores únicos")
print("\nLista de vendedores:")
print(df_vendedores.head(10))

👨‍💼 TABLA VENDEDORES
• 30 vendedores únicos

Lista de vendedores:
   VendedorID             NombreVendedor
0           1      Evaristo Guillen Peña
1           2  Poncio Gabriel Poza Acedo
2           3   Herminia Meléndez-Huguet
3           4          Aura del Bermúdez
4           5  María Belén Alegria Camps
5           6       Eloísa Plaza Hurtado
6           7           Edu Juan Pedraza
7           8    Griselda Arnaiz Palacio
8           9         Olivia Armas Ayuso
9          10        Armida Azorin Plaza


### 4.6 Función de Limpieza de Precios

In [26]:
def limpiar_precio_mejorado(precio):
    """
    Función robusta para limpiar precios en cualquier formato
    """
    if pd.isna(precio):
        return np.nan
    
    # Si ya es numérico, devolver directamente
    if isinstance(precio, (int, float)):
        return float(precio)
    
    # Si es texto, limpiarlo
    if isinstance(precio, str):
        # Remover todos los caracteres no numéricos excepto punto decimal
        precio_limpio = re.sub(r'[^\d.]', '', precio.strip())
        if precio_limpio and precio_limpio != '.':
            try:
                return float(precio_limpio)
            except:
                return np.nan
    return np.nan

print("✅ Función de limpieza de precios mejorada creada")

✅ Función de limpieza de precios mejorada creada


### 4.7 Tabla Productos (con precios limpios)

In [27]:
print("🛍️ CREANDO TABLA PRODUCTOS CON PRECIOS LIMPIOS")
print("=" * 50)

# Crear dataset limpio para productos
df_ventas_limpio = df_ventas.copy()

# Limpiar precios en el dataset
columnas_precio = ['Producto1PrecioUnitario', 'Producto2PrecioUnitario', 'Producto3PrecioUnitario',
                   'Producto1Subtotal', 'Producto2Subtotal', 'Producto3Subtotal']

print("🧹 Limpiando precios en columnas...")
for columna in columnas_precio:
    if columna in df_ventas_limpio.columns:
        df_ventas_limpio[columna] = df_ventas_limpio[columna].apply(limpiar_precio_mejorado)
        print(f"  • {columna}: {df_ventas_limpio[columna].notna().sum()} valores válidos")
    else:
        print(f"  • {columna}: COLUMNA NO ENCONTRADA")

# Consolidar productos únicos
productos_list = []

# Producto 1
if 'Producto1Nombre' in df_ventas_limpio.columns:
    productos_df1 = df_ventas_limpio[['Producto1Nombre', 'Producto1Marca', 'Producto1PrecioUnitario']].dropna(subset=['Producto1Nombre'])
    productos_df1.columns = ['NombreProducto', 'Marca', 'PrecioUnitario']
    productos_list.append(productos_df1)

# Producto 2
if 'Producto2Nombre' in df_ventas_limpio.columns:
    productos_df2 = df_ventas_limpio[['Producto2Nombre', 'Producto2Marca', 'Producto2PrecioUnitario']].dropna(subset=['Producto2Nombre'])
    productos_df2.columns = ['NombreProducto', 'Marca', 'PrecioUnitario']
    productos_list.append(productos_df2)

# Producto 3
if 'Producto3Nombre' in df_ventas_limpio.columns:
    productos_df3 = df_ventas_limpio[['Producto3Nombre', 'Producto3Marca', 'Producto3PrecioUnitario']].dropna(subset=['Producto3Nombre'])
    productos_df3.columns = ['NombreProducto', 'Marca', 'PrecioUnitario']
    productos_list.append(productos_df3)

# Combinar y eliminar duplicados
if productos_list:
    df_todos_productos = pd.concat(productos_list, ignore_index=True).drop_duplicates()
    df_productos = df_todos_productos.reset_index(drop=True)
    df_productos['ProductoID'] = range(1, len(df_productos) + 1)
    df_productos = df_productos[['ProductoID', 'NombreProducto', 'Marca', 'PrecioUnitario']]
else:
    print("❌ No se encontraron columnas de productos")
    df_productos = pd.DataFrame(columns=['ProductoID', 'NombreProducto', 'Marca', 'PrecioUnitario'])

print(f"\n✅ TABLA PRODUCTOS: {len(df_productos)} productos únicos")
if len(df_productos) > 0:
    print(f"• Productos con precios válidos: {df_productos['PrecioUnitario'].notna().sum()}/{len(df_productos)}")
    print("\n📋 Muestra de productos:")
    print(df_productos.head(10))

🛍️ CREANDO TABLA PRODUCTOS CON PRECIOS LIMPIOS
🧹 Limpiando precios en columnas...
  • Producto1PrecioUnitario: 30000 valores válidos
  • Producto2PrecioUnitario: 30000 valores válidos
  • Producto3PrecioUnitario: 30000 valores válidos
  • Producto1Subtotal: 30000 valores válidos
  • Producto2Subtotal: 30000 valores válidos
  • Producto3Subtotal: 30000 valores válidos

✅ TABLA PRODUCTOS: 80 productos únicos
• Productos con precios válidos: 80/80

📋 Muestra de productos:
   ProductoID             NombreProducto   Marca  PrecioUnitario
0           1         Dell Alienware m15    Dell       8000000.0
1           2            HP Spectre x360      HP       5200000.0
2           3           Lenovo IdeaPad 5  Lenovo       2800000.0
3           4               Acer Swift 3    Acer       2600000.0
4           5  Lenovo ThinkPad X1 Carbon  Lenovo       6800000.0
5           6           Dell Inspiron 15    Dell       3000000.0
6           7              Acer Aspire 5    Acer       2000000.0
7     

### FASE 5: Creación de Tablas de Hechos
#### 5.1 Tabla Facturas

In [28]:
print("🧾 CREANDO TABLA FACTURAS")
print("=" * 40)

# Preparar datos para facturas
columnas_necesarias = ['VentaID', 'Año', 'MesNumero', 'Día', 'SucursalNombre', 'ClienteNombre', 'VendedorNombre']
columnas_faltantes = [col for col in columnas_necesarias if col not in df_ventas.columns]

if columnas_faltantes:
    print(f"⚠️  Columnas faltantes: {columnas_faltantes}")
    # Usar las columnas disponibles
    columnas_disponibles = [col for col in columnas_necesarias if col in df_ventas.columns]
    df_facturas = df_ventas[columnas_disponibles].copy()
else:
    df_facturas = df_ventas[['VentaID', 'Año', 'MesNumero', 'Día', 'SucursalNombre', 'ClienteNombre', 'VendedorNombre', 'MetodoPago', 'DescuentoVenta']].copy()

# Crear fecha completa si tenemos las columnas necesarias
if all(col in df_facturas.columns for col in ['Año', 'MesNumero', 'Día']):
    df_facturas['FechaVenta'] = pd.to_datetime(
        df_facturas['Año'].astype(str) + '-' + 
        df_facturas['MesNumero'].astype(str) + '-' + 
        df_facturas['Día'].astype(str)
    )
else:
    print("⚠️  No se pueden crear fechas, columnas faltantes")
    df_facturas['FechaVenta'] = pd.Timestamp.now()

# Agregar hora (usando datos disponibles o valor por defecto)
df_facturas['HoraVenta'] = '12:00:00'  # Valor por defecto

# Mapear IDs con manejo de errores
sucursal_map = dict(zip(df_sucursales['SucursalNombre'], df_sucursales['SucursalID']))
cliente_map = dict(zip(df_clientes['Nombre'], df_clientes['ClienteID']))
vendedor_map = dict(zip(df_vendedores['NombreVendedor'], df_vendedores['VendedorID']))

def mapear_seguro(valor, mapeo, default=1):
    """Mapea valores de forma segura"""
    if pd.isna(valor):
        return default
    return mapeo.get(valor, default)

if 'SucursalNombre' in df_facturas.columns:
    df_facturas['SucursalID'] = df_facturas['SucursalNombre'].apply(lambda x: mapear_seguro(x, sucursal_map))
if 'ClienteNombre' in df_facturas.columns:
    df_facturas['ClienteID'] = df_facturas['ClienteNombre'].apply(lambda x: mapear_seguro(x, cliente_map))
if 'VendedorNombre' in df_facturas.columns:
    df_facturas['VendedorID'] = df_facturas['VendedorNombre'].apply(lambda x: mapear_seguro(x, vendedor_map))

# Calcular TotalVenta
def calcular_total_venta(index):
    total = 0.0
    for i in range(1, 4):
        subtotal_col = f'Producto{i}Subtotal'
        if subtotal_col in df_ventas_limpio.columns:
            valor = df_ventas_limpio.loc[index, subtotal_col]
            if pd.notna(valor):
                total += valor
    return total

print("💰 Calculando totales de venta...")
df_facturas['TotalVenta'] = [calcular_total_venta(i) for i in range(len(df_facturas))]

# Seleccionar columnas finales
columnas_finales = ['VentaID', 'FechaVenta', 'HoraVenta', 'ClienteID', 'SucursalID', 'VendedorID']
if 'MetodoPago' in df_facturas.columns:
    columnas_finales.append('MetodoPago')
if 'DescuentoVenta' in df_facturas.columns:
    columnas_finales.append('DescuentoVenta')
columnas_finales.append('TotalVenta')

df_facturas = df_facturas[columnas_finales]

# Renombrar columnas
rename_dict = {
    'VentaID': 'FacturaID',
    'DescuentoVenta': 'Descuento'
}
df_facturas = df_facturas.rename(columns=rename_dict)

print(f"✅ TABLA FACTURAS: {len(df_facturas)} facturas creadas")
print(f"• Total ventas: ${df_facturas['TotalVenta'].sum():,.0f}")
if 'MetodoPago' in df_facturas.columns:
    print(f"• Métodos de pago únicos: {df_facturas['MetodoPago'].nunique()}")

print("\n📄 Muestra de facturas:")
print(df_facturas.head(5))

🧾 CREANDO TABLA FACTURAS
💰 Calculando totales de venta...
✅ TABLA FACTURAS: 30000 facturas creadas
• Total ventas: $2,572,205,600,000
• Métodos de pago únicos: 6

📄 Muestra de facturas:
   FacturaID FechaVenta HoraVenta  ClienteID  SucursalID  VendedorID  \
0       7386 2016-12-21  12:00:00          1           1           1   
1      28232 2018-12-06  12:00:00          2           1           2   
2      11989 2021-02-02  12:00:00          3           2           3   
3      19343 2025-05-13  12:00:00          4           1           4   
4       3666 2020-12-27  12:00:00          5           1           5   

        MetodoPago  Descuento  TotalVenta  
0  No especificado          0   8000000.0  
1  Tarjeta Crédito          1   5200000.0  
2  Tarjeta Crédito          0   5200000.0  
3         Efectivo          0   2800000.0  
4  Tarjeta Crédito          1   5200000.0  


### 5.2 Tabla DetalleFacturas

In [29]:
print("📝 CREANDO TABLA DETALLEFACTURAS")
print("=" * 45)

# Crear mapeo de productos
producto_map = {}
for _, row in df_productos.iterrows():
    clave = (row['NombreProducto'], row['Marca'])
    producto_map[clave] = row['ProductoID']

print(f"🗂️ Mapeo de productos creado: {len(producto_map)} productos")

# Procesar líneas de detalle
detalles_list = []
detalle_id = 1

print("🔍 Procesando líneas de detalle...")
for index, factura in df_facturas.iterrows():
    factura_id = factura['FacturaID']
    
    # Procesar Producto 1
    if 'Producto1Nombre' in df_ventas.columns and pd.notna(df_ventas.loc[index, 'Producto1Nombre']):
        clave_producto = (df_ventas.loc[index, 'Producto1Nombre'], df_ventas.loc[index, 'Producto1Marca'])
        producto_id = producto_map.get(clave_producto)
        
        if producto_id:
            detalles_list.append({
                'DetalleID': detalle_id,
                'FacturaID': factura_id,
                'ProductoID': producto_id,
                'Cantidad': df_ventas_limpio.loc[index, 'Producto1Cantidad'],
                'PrecioUnitario': df_ventas_limpio.loc[index, 'Producto1PrecioUnitario'],
                'Subtotal': df_ventas_limpio.loc[index, 'Producto1Subtotal']
            })
            detalle_id += 1
    
    # Procesar Producto 2
    if 'Producto2Nombre' in df_ventas.columns and pd.notna(df_ventas.loc[index, 'Producto2Nombre']):
        clave_producto = (df_ventas.loc[index, 'Producto2Nombre'], df_ventas.loc[index, 'Producto2Marca'])
        producto_id = producto_map.get(clave_producto)
        
        if producto_id:
            detalles_list.append({
                'DetalleID': detalle_id,
                'FacturaID': factura_id,
                'ProductoID': producto_id,
                'Cantidad': df_ventas_limpio.loc[index, 'Producto2Cantidad'],
                'PrecioUnitario': df_ventas_limpio.loc[index, 'Producto2PrecioUnitario'],
                'Subtotal': df_ventas_limpio.loc[index, 'Producto2Subtotal']
            })
            detalle_id += 1
    
    # Procesar Producto 3
    if 'Producto3Nombre' in df_ventas.columns and pd.notna(df_ventas.loc[index, 'Producto3Nombre']):
        clave_producto = (df_ventas.loc[index, 'Producto3Nombre'], df_ventas.loc[index, 'Producto3Marca'])
        producto_id = producto_map.get(clave_producto)
        
        if producto_id:
            detalles_list.append({
                'DetalleID': detalle_id,
                'FacturaID': factura_id,
                'ProductoID': producto_id,
                'Cantidad': df_ventas_limpio.loc[index, 'Producto3Cantidad'],
                'PrecioUnitario': df_ventas_limpio.loc[index, 'Producto3PrecioUnitario'],
                'Subtotal': df_ventas_limpio.loc[index, 'Producto3Subtotal']
            })
            detalle_id += 1

# Crear DataFrame final
if detalles_list:
    df_detalle_facturas = pd.DataFrame(detalles_list)
    
    # Asegurar tipos de datos numéricos
    df_detalle_facturas['Cantidad'] = pd.to_numeric(df_detalle_facturas['Cantidad'], errors='coerce')
    df_detalle_facturas['PrecioUnitario'] = pd.to_numeric(df_detalle_facturas['PrecioUnitario'], errors='coerce')
    df_detalle_facturas['Subtotal'] = pd.to_numeric(df_detalle_facturas['Subtotal'], errors='coerce')
    
    print(f"✅ TABLA DETALLEFACTURAS: {len(df_detalle_facturas)} líneas de detalle")
    print(f"• Facturas con detalles: {df_detalle_facturas['FacturaID'].nunique()}")
    print(f"• Productos vendidos: {df_detalle_facturas['ProductoID'].nunique()}")
    print(f"• Total unidades vendidas: {df_detalle_facturas['Cantidad'].sum():,.0f}")
    
    print("\n📋 Muestra de detalles:")
    print(df_detalle_facturas.head(10))
else:
    print("❌ No se pudieron crear líneas de detalle")
    df_detalle_facturas = pd.DataFrame(columns=['DetalleID', 'FacturaID', 'ProductoID', 'Cantidad', 'PrecioUnitario', 'Subtotal'])

📝 CREANDO TABLA DETALLEFACTURAS
🗂️ Mapeo de productos creado: 40 productos
🔍 Procesando líneas de detalle...
✅ TABLA DETALLEFACTURAS: 60059 líneas de detalle
• Facturas con detalles: 30000
• Productos vendidos: 40
• Total unidades vendidas: 495,597

📋 Muestra de detalles:
   DetalleID  FacturaID  ProductoID  Cantidad  PrecioUnitario   Subtotal
0          1       7386          57         1       8000000.0  8000000.0
1          2      28232          43         1       5200000.0  5200000.0
2          3      11989          43         1       5200000.0  5200000.0
3          4      19343          63         1       2800000.0  2800000.0
4          5       3666          43         1       5200000.0  5200000.0
5          6       1012          47         1       2600000.0  2600000.0
6          7      26419          41         1       6800000.0  6800000.0
7          8      27840          46         1       3000000.0  3000000.0
8          9      20290          62         1       2000000.0  2000000

### Correccion De Ciudades A Sucursales

In [30]:
print("🔧 CORRIGIENDO ASIGNACIÓN DE CIUDADES A SUCURSALES")
print("=" * 55)

# Función corregida para asignar ciudades
def asignar_ciudad_corregida(sucursal_nombre):
    """
    Asigna ciudad CORRECTAMENTE basada en el nombre de sucursal
    """
    sucursal_lower = str(sucursal_nombre).lower()
    
    if 'medellín' in sucursal_lower or 'medellin' in sucursal_lower:
        return 'Medellín'
    elif 'bogotá' in sucursal_lower or 'bogota' in sucursal_lower:
        return 'Bogotá'
    elif 'cali' in sucursal_lower:
        return 'Cali'
    elif 'pereira' in sucursal_lower:
        return 'Pereira'
    else:
        return 'No especificado'

print("🔄 Re-asignando ciudades a sucursales...")

# Crear una nueva tabla de sucursales con las ciudades correctas
sucursales_corregidas = []
for _, sucursal in df_sucursales.iterrows():
    ciudad_corregida = asignar_ciudad_corregida(sucursal['SucursalNombre'])
    
    sucursales_corregidas.append({
        'SucursalID': sucursal['SucursalID'],
        'SucursalNombre': sucursal['SucursalNombre'],
        'CiudadAsignada': ciudad_corregida
    })

df_sucursales_corregido = pd.DataFrame(sucursales_corregidas)

# Mapear CiudadID correctamente
ciudad_map_corregido = {
    'Bogotá': 1,
    'Cali': 2, 
    'Medellín': 3,
    'Pereira': 5,
    'No especificado': 4
}

df_sucursales_corregido['CiudadID'] = df_sucursales_corregido['CiudadAsignada'].map(ciudad_map_corregido)

# Reemplazar la tabla original
df_sucursales = df_sucursales_corregido[['SucursalID', 'SucursalNombre', 'CiudadID']]

print("✅ TABLA SUCURSALES CORREGIDA:")
print(df_sucursales)

print(f"\n🏙️  DISTRIBUCIÓN CORREGIDA SUCURSALES -> CIUDADES:")
sucursales_con_ciudad = df_sucursales.merge(df_ciudades, on='CiudadID', how='left')
for _, row in sucursales_con_ciudad.iterrows():
    print(f"  • {row['SucursalNombre']:25} -> CiudadID: {row['CiudadID']} -> {row['NombreCiudad']}")

# %% [markdown]
# ### ACTUALIZAR FACTURAS CON LAS CIUDADES CORREGIDAS

# %%
print("\n🔄 ACTUALIZANDO FACTURAS CON CIUDADES CORREGIDAS")
print("=" * 50)

# Actualizar el mapeo SucursalID -> CiudadID
sucursal_ciudad_map_corregido = dict(zip(df_sucursales['SucursalID'], df_sucursales['CiudadID']))
print(f"📋 Nuevo mapeo SucursalID -> CiudadID: {sucursal_ciudad_map_corregido}")

# Aplicar el nuevo mapeo a las facturas
df_facturas['CiudadID'] = df_facturas['SucursalID'].map(sucursal_ciudad_map_corregido)

# Verificar los resultados
print(f"\n✅ DISTRIBUCIÓN FINAL DE CIUDADES EN FACTURAS:")
ciudad_counts_final = df_facturas['CiudadID'].value_counts().sort_index()
for ciudad_id, count in ciudad_counts_final.items():
    ciudad_info = df_ciudades[df_ciudades['CiudadID'] == ciudad_id]
    if len(ciudad_info) > 0:
        ciudad_nombre = ciudad_info['NombreCiudad'].values[0]
    else:
        ciudad_nombre = "No encontrada"
    print(f"  • CiudadID {ciudad_id} ({ciudad_nombre}): {count} facturas")

# Mostrar distribución detallada por sucursal y ciudad
print(f"\n🔍 DISTRIBUCIÓN DETALLADA SUCURSAL -> CIUDAD:")
distribucion_detallada = df_facturas.merge(
    df_sucursales[['SucursalID', 'SucursalNombre']], 
    on='SucursalID', 
    how='left'
).merge(
    df_ciudades[['CiudadID', 'NombreCiudad']], 
    on='CiudadID', 
    how='left'
)

agrupado = distribucion_detallada.groupby(['SucursalNombre', 'NombreCiudad']).agg({
    'FacturaID': 'count',
    'TotalVenta': 'sum'
}).reset_index()

print("\n📊 VENTAS POR SUCURSAL Y CIUDAD:")
for _, row in agrupado.iterrows():
    print(f"  • {row['SucursalNombre']:25} -> {row['NombreCiudad']:10}: {row['FacturaID']:>3} facturas, ${row['TotalVenta']:>12,.0f}")

# Verificar específicamente Medellín
print(f"\n📍 VERIFICACIÓN ESPECÍFICA DE MEDELLÍN:")
facturas_medellin = distribucion_detallada[distribucion_detallada['NombreCiudad'] == 'Medellín']
if len(facturas_medellin) > 0:
    print("✅ Sucursales de Medellín correctamente identificadas:")
    medellin_agrupado = facturas_medellin.groupby('SucursalNombre')['FacturaID'].count()
    for sucursal, count in medellin_agrupado.items():
        print(f"  • {sucursal}: {count} facturas")
else:
    print("❌ No se encontraron facturas de Medellín")

# Verificar específicamente Bogotá
print(f"\n📍 VERIFICACIÓN ESPECÍFICA DE BOGOTÁ:")
facturas_bogota = distribucion_detallada[distribucion_detallada['NombreCiudad'] == 'Bogotá']
if len(facturas_bogota) > 0:
    print("✅ Sucursales de Bogotá correctamente identificadas:")
    bogota_agrupado = facturas_bogota.groupby('SucursalNombre')['FacturaID'].count()
    for sucursal, count in bogota_agrupado.items():
        print(f"  • {sucursal}: {count} facturas")
else:
    print("❌ No se encontraron facturas de Bogotá")

# %% [markdown]
# ### ACTUALIZAR EL ARCHIVO EXCEL CON LOS DATOS CORREGIDOS

# %%
print("\n💾 ACTUALIZANDO ARCHIVO EXCEL CON DATOS CORREGIDOS")
print("=" * 55)

try:
    with pd.ExcelWriter('modeloVentas_corregido.xlsx', engine='openpyxl') as writer:
        # Tablas corregidas
        df_facturas.to_excel(writer, sheet_name='Facturas', index=False)
        df_sucursales.to_excel(writer, sheet_name='Sucursales', index=False)
        
        # Otras tablas
        if 'df_detalle_facturas' in locals() and len(df_detalle_facturas) > 0:
            df_detalle_facturas.to_excel(writer, sheet_name='DetalleFacturas', index=False)
        df_clientes.to_excel(writer, sheet_name='Clientes', index=False)
        df_ciudades.to_excel(writer, sheet_name='Ciudades', index=False)
        df_vendedores.to_excel(writer, sheet_name='Vendedores', index=False)
        if 'df_productos' in locals() and len(df_productos) > 0:
            df_productos.to_excel(writer, sheet_name='Productos', index=False)

    print("✅ ARCHIVO CORREGIDO GUARDADO: modeloVentas_corregido.xlsx")
    
    # Mostrar resumen final
    print(f"\n🎯 RESUMEN FINAL CORREGIDO:")
    print("=" * 40)
    print("🏙️  DISTRIBUCIÓN POR CIUDAD:")
    for ciudad_id in sorted(df_facturas['CiudadID'].unique()):
        count = (df_facturas['CiudadID'] == ciudad_id).sum()
        ciudad_nombre = df_ciudades[df_ciudades['CiudadID'] == ciudad_id]['NombreCiudad'].values[0]
        ventas = df_facturas[df_facturas['CiudadID'] == ciudad_id]['TotalVenta'].sum()
        print(f"  • {ciudad_nombre:10} (ID: {ciudad_id}): {count:>4} facturas, ${ventas:>12,.0f}")
        
except Exception as e:
    print(f"❌ Error guardando archivo corregido: {e}")

🔧 CORRIGIENDO ASIGNACIÓN DE CIUDADES A SUCURSALES
🔄 Re-asignando ciudades a sucursales...
✅ TABLA SUCURSALES CORREGIDA:
   SucursalID        SucursalNombre  CiudadID
0           1    TechCore Bogotá #1         1
1           2    TechCore Bogotá #2         1
2           3         TechCore Cali         2
3           4  TechCore Medellín #2         3
4           5  TechCore Medellín #1         3
5           6      TechCore Pereira         5

🏙️  DISTRIBUCIÓN CORREGIDA SUCURSALES -> CIUDADES:
  • TechCore Bogotá #1        -> CiudadID: 1 -> Bogota
  • TechCore Bogotá #2        -> CiudadID: 1 -> Bogota
  • TechCore Cali             -> CiudadID: 2 -> Cali
  • TechCore Medellín #2      -> CiudadID: 3 -> Medellin
  • TechCore Medellín #1      -> CiudadID: 3 -> Medellin
  • TechCore Pereira          -> CiudadID: 5 -> Pereira

🔄 ACTUALIZANDO FACTURAS CON CIUDADES CORREGIDAS
📋 Nuevo mapeo SucursalID -> CiudadID: {1: 1, 2: 1, 3: 2, 4: 3, 5: 3, 6: 5}

✅ DISTRIBUCIÓN FINAL DE CIUDADES EN FACTURAS:
  

### FASE 6: Validaciones y Reportes
### 6.1 Validaciones de Integridad Referencial

In [31]:
print("🔍 VALIDACIONES DE INTEGRIDAD REFERENCIAL")
print("=" * 50)

# Solo realizar validaciones si tenemos datos
if len(df_detalle_facturas) > 0:
    # Validación de Facturas en DetalleFacturas
    facturas_en_detalle = set(df_detalle_facturas['FacturaID'])
    facturas_existentes = set(df_facturas['FacturaID'])
    facturas_huérfanas = facturas_en_detalle - facturas_existentes
    print(f"✓ Facturas huérfanas en DetalleFacturas: {len(facturas_huérfanas)}")

    # Validación de Productos en DetalleFacturas
    productos_en_detalle = set(df_detalle_facturas['ProductoID'])
    productos_existentes = set(df_productos['ProductoID'])
    productos_huérfanos = productos_en_detalle - productos_existentes
    print(f"✓ Productos huérfanos en DetalleFacturas: {len(productos_huérfanos)}")
else:
    print("⚠️  No hay datos en DetalleFacturas para validar")

# Validación de Clientes en Facturas
if 'ClienteID' in df_facturas.columns:
    clientes_en_facturas = set(df_facturas['ClienteID'])
    clientes_existentes = set(df_clientes['ClienteID'])
    clientes_huérfanos = clientes_en_facturas - clientes_existentes
    print(f"✓ Clientes huérfanos en Facturas: {len(clientes_huérfanos)}")

# Validación de Sucursales en Facturas
if 'SucursalID' in df_facturas.columns:
    sucursales_en_facturas = set(df_facturas['SucursalID'])
    sucursales_existentes = set(df_sucursales['SucursalID'])
    sucursales_huérfanas = sucursales_en_facturas - sucursales_existentes
    print(f"✓ Sucursales huérfanas en Facturas: {len(sucursales_huérfanas)}")

# Validación de Vendedores en Facturas
if 'VendedorID' in df_facturas.columns:
    vendedores_en_facturas = set(df_facturas['VendedorID'])
    vendedores_existentes = set(df_vendedores['VendedorID'])
    vendedores_huérfanos = vendedores_en_facturas - vendedores_existentes
    print(f"✓ Vendedores huérfanos en Facturas: {len(vendedores_huérfanos)}")

print("\n🎉 VALIDACIONES COMPLETADAS")

🔍 VALIDACIONES DE INTEGRIDAD REFERENCIAL
✓ Facturas huérfanas en DetalleFacturas: 0
✓ Productos huérfanos en DetalleFacturas: 0
✓ Clientes huérfanos en Facturas: 0
✓ Sucursales huérfanas en Facturas: 0
✓ Vendedores huérfanos en Facturas: 0

🎉 VALIDACIONES COMPLETADAS


### 6.2 Reportes Exploratorios

In [33]:
print("📈 REPORTES EXPLORATORIOS")
print("=" * 40)

if len(df_detalle_facturas) > 0 and len(df_productos) > 0:
    # Total de ventas por marca
    ventas_por_marca = df_detalle_facturas.merge(df_productos, on='ProductoID')
    total_ventas_marca = ventas_por_marca.groupby('Marca')['Subtotal'].sum().sort_values(ascending=False)

    print("🏷️ TOTAL DE VENTAS POR MARCA:")
    print("-" * 30)
    for i, (marca, total) in enumerate(total_ventas_marca.head(10).items(), 1):
        print(f"  {i:2d}. {marca:15} ${total:>15,.0f}")

    # Top productos más vendidos
    productos_vendidos = ventas_por_marca.groupby(['NombreProducto', 'Marca']).agg({
        'Cantidad': 'sum',
        'Subtotal': 'sum'
    }).sort_values('Cantidad', ascending=False)

    print(f"\n🏆 TOP 10 PRODUCTOS MÁS VENDIDOS (por cantidad):")
    print("-" * 50)
    for i, (producto, datos) in enumerate(productos_vendidos.head(10).iterrows(), 1):
        nombre, marca = producto
        print(f"  {i:2d}. {nombre:25} ({marca:10}) {datos['Cantidad']:>4} unidades - ${datos['Subtotal']:>12,.0f}")
else:
    print("⚠️  No hay suficientes datos para generar reportes de productos")

# Análisis por sucursal - CORREGIDO
if 'SucursalID' in df_facturas.columns:
    print(f"\n🔍 VERIFICANDO COLUMNAS DISPONIBLES PARA ANÁLISIS DE SUCURSALES:")
    print(f"   Columnas en df_facturas: {list(df_facturas.columns)}")
    print(f"   Columnas en df_sucursales: {list(df_sucursales.columns)}")
    print(f"   Columnas en df_ciudades: {list(df_ciudades.columns)}")
    
    try:
        # Hacer el merge paso a paso para debuggear
        print(f"\n🔄 REALIZANDO MERGE DE TABLAS...")
        
        # Merge 1: Facturas con Sucursales
        ventas_con_sucursales = df_facturas.merge(df_sucursales, on='SucursalID')
        print(f"   ✅ Merge con sucursales: {len(ventas_con_sucursales)} registros")
        print(f"   Columnas después del primer merge: {list(ventas_con_sucursales.columns)}")
        
        # Merge 2: Con Ciudades
        ventas_por_sucursal = ventas_con_sucursales.merge(df_ciudades, on='CiudadID')
        print(f"   ✅ Merge con ciudades: {len(ventas_por_sucursal)} registros")
        print(f"   Columnas después del segundo merge: {list(ventas_por_sucursal.columns)}")
        
        # Verificar que las columnas necesarias existen
        columnas_necesarias = ['SucursalNombre', 'NombreCiudad', 'TotalVenta']
        columnas_faltantes = [col for col in columnas_necesarias if col not in ventas_por_sucursal.columns]
        
        if columnas_faltantes:
            print(f"   ⚠️  Columnas faltantes: {columnas_faltantes}")
            print(f"   🎯 Columnas disponibles: {[col for col in ventas_por_sucursal.columns if 'sucursal' in col.lower() or 'ciudad' in col.lower() or 'nombre' in col.lower()]}")
            
            # Intentar con nombres alternativos
            if 'SucursalNombre' not in ventas_por_sucursal.columns:
                # Buscar columnas que contengan "sucursal"
                sucursal_cols = [col for col in ventas_por_sucursal.columns if 'sucursal' in col.lower()]
                if sucursal_cols:
                    col_sucursal = sucursal_cols[0]
                    print(f"   🔄 Usando columna alternativa para sucursal: {col_sucursal}")
                else:
                    col_sucursal = 'SucursalID'
                    print(f"   ⚠️  Usando ID de sucursal por defecto")
            else:
                col_sucursal = 'SucursalNombre'
                
            if 'NombreCiudad' not in ventas_por_sucursal.columns:
                # Buscar columnas que contengan "ciudad"
                ciudad_cols = [col for col in ventas_por_sucursal.columns if 'ciudad' in col.lower()]
                if ciudad_cols:
                    col_ciudad = ciudad_cols[0]
                    print(f"   🔄 Usando columna alternativa para ciudad: {col_ciudad}")
                else:
                    col_ciudad = 'CiudadID'
                    print(f"   ⚠️  Usando ID de ciudad por defecto")
            else:
                col_ciudad = 'NombreCiudad'
                
            # Agrupar con columnas alternativas
            ventas_sucursal_agrupado = ventas_por_sucursal.groupby([col_sucursal, col_ciudad])['TotalVenta'].sum().sort_values(ascending=False)
            
        else:
            # Agrupar con columnas originales
            ventas_sucursal_agrupado = ventas_por_sucursal.groupby(['SucursalNombre', 'NombreCiudad'])['TotalVenta'].sum().sort_values(ascending=False)
        
        print(f"\n🏪 VENTAS POR SUCURSAL:")
        print("-" * 50)
        
        if len(ventas_sucursal_agrupado) > 0:
            for i, ((sucursal, ciudad), total) in enumerate(ventas_sucursal_agrupado.head(10).items(), 1):
                print(f"  {i:2d}. {str(sucursal)[:25]:25} ({str(ciudad)[:12]:12}) ${total:>12,.0f}")
        else:
            print("  ⚠️  No hay datos para mostrar")
            
    except Exception as e:
        print(f"❌ Error en análisis de sucursales: {e}")
        print(f"   💡 Intentando análisis alternativo...")
        
        # Análisis alternativo simple
        try:
            ventas_simple = df_facturas.groupby('SucursalID')['TotalVenta'].sum().sort_values(ascending=False)
            print(f"\n🏪 VENTAS POR SUCURSAL (ID only):")
            print("-" * 35)
            for sucursal_id, total in ventas_simple.head(10).items():
                # Buscar nombre de sucursal
                sucursal_nombre = df_sucursales[df_sucursales['SucursalID'] == sucursal_id]['SucursalNombre']
                if len(sucursal_nombre) > 0:
                    nombre = sucursal_nombre.values[0]
                else:
                    nombre = f"Sucursal {sucursal_id}"
                print(f"  • {nombre:25} ${total:>12,.0f}")
        except Exception as e2:
            print(f"❌ Error en análisis alternativo: {e2}")

else:
    print("⚠️  No hay datos de sucursales para analizar")

📈 REPORTES EXPLORATORIOS
🏷️ TOTAL DE VENTAS POR MARCA:
------------------------------
   1. Lenovo          $656,165,600,000
   2. HP              $529,810,000,000
   3. Dell            $476,146,600,000
   4. Apple           $422,534,400,000
   5. Asus            $182,941,400,000
   6. Acer            $121,788,800,000
   7. Samsung         $ 67,596,000,000
   8. MSI             $ 47,754,400,000
   9. Microsoft       $ 45,193,600,000
  10. Razer           $ 22,274,800,000

🏆 TOP 10 PRODUCTOS MÁS VENDIDOS (por cantidad):
--------------------------------------------------
   1. HP Spectre x360           (HP        ) 41350.0 unidades - $215,020,000,000
   2. Lenovo ThinkPad X1 Carbon (Lenovo    ) 32724.0 unidades - $222,523,200,000
   3. Lenovo Legion 5 Pro       (Lenovo    ) 32033.0 unidades - $230,637,600,000
   4. Lenovo Yoga 7i            (Lenovo    ) 28605.0 unidades - $125,862,000,000
   5. HP Omen 16                (HP        ) 27781.0 unidades - $166,686,000,000
   6. Lenovo IdeaPa

### FASE 7: Exportación Final

In [34]:
print("💾 EXPORTACIÓN FINAL DEL MODELO RELACIONAL")
print("=" * 50)

# Exportar a Excel
archivo_salida = 'ModeloVentas.xlsx'

try:
    with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
        # Tablas de hechos
        df_facturas.to_excel(writer, sheet_name='Facturas', index=False)
        if len(df_detalle_facturas) > 0:
            df_detalle_facturas.to_excel(writer, sheet_name='DetalleFacturas', index=False)
        
        # Tablas dimensión
        if len(df_productos) > 0:
            df_productos.to_excel(writer, sheet_name='Productos', index=False)
        df_clientes.to_excel(writer, sheet_name='Clientes', index=False)
        df_sucursales.to_excel(writer, sheet_name='Sucursales', index=False)
        df_ciudades.to_excel(writer, sheet_name='Ciudades', index=False)
        df_vendedores.to_excel(writer, sheet_name='Vendedores', index=False)

    # Verificar archivo creado
    if os.path.exists(archivo_salida):
        file_size = os.path.getsize(archivo_salida) / 1024
        print(f"✅ ARCHIVO CREADO EXITOSAMENTE: {archivo_salida}")
        print(f"📏 Tamaño del archivo: {file_size:.1f} KB")
        
        # Mostrar resumen de hojas
        excel_file = pd.ExcelFile(archivo_salida)
        print(f"📑 Hojas en el archivo: {excel_file.sheet_names}")
        
        # Mostrar resumen de registros por hoja
        print("\n📊 RESUMEN DE REGISTROS POR HOJA:")
        for sheet in excel_file.sheet_names:
            df_temp = pd.read_excel(archivo_salida, sheet_name=sheet)
            print(f"  • {sheet:15} : {len(df_temp):>5} registros")
    else:
        print("❌ Error: No se pudo crear el archivo")
        
except Exception as e:
    print(f"❌ Error durante la exportación: {e}")

💾 EXPORTACIÓN FINAL DEL MODELO RELACIONAL
✅ ARCHIVO CREADO EXITOSAMENTE: ModeloVentas.xlsx
📏 Tamaño del archivo: 4206.8 KB
📑 Hojas en el archivo: ['Facturas', 'DetalleFacturas', 'Productos', 'Clientes', 'Sucursales', 'Ciudades', 'Vendedores']

📊 RESUMEN DE REGISTROS POR HOJA:
  • Facturas        : 30000 registros
  • DetalleFacturas : 60059 registros
  • Productos       :    80 registros
  • Clientes        : 17449 registros
  • Sucursales      :     6 registros
  • Ciudades        :     5 registros
  • Vendedores      :    30 registros


### FASE 8: Resumen Final

In [35]:
print("🎯 RESUMEN FINAL - AVANCE 2 COMPLETADO")
print("=" * 60)
print("MODELO RELACIONAL CREADO EXITOSAMENTE")
print("=" * 60)

# Resumen de tablas creadas
resumen_tablas = {
    'Ciudades': len(df_ciudades),
    'Sucursales': len(df_sucursales),
    'Clientes': len(df_clientes),
    'Vendedores': len(df_vendedores),
    'Productos': len(df_productos),
    'Facturas': len(df_facturas),
    'DetalleFacturas': len(df_detalle_facturas)
}

print("📋 RESUMEN DE TABLAS CREADAS:")
for tabla, cantidad in resumen_tablas.items():
    print(f"  • {tabla:15} : {cantidad:>5} registros")

print("\n💰 MÉTRICAS PRINCIPALES:")
print(f"  • Ventas totales: ${df_facturas['TotalVenta'].sum():,.0f}")
if len(df_detalle_facturas) > 0:
    print(f"  • Unidades vendidas: {df_detalle_facturas['Cantidad'].sum():,.0f}")
    print(f"  • Productos únicos vendidos: {df_detalle_facturas['ProductoID'].nunique()}")
print(f"  • Facturas procesadas: {len(df_facturas)}")
print(f"  • Líneas de detalle: {len(df_detalle_facturas)}")

print("\n🔄 MEJORAS INCORPORADAS:")
mejoras = [
    "Lógica inteligente para asignación ciudad-sucursal",
    "Manejo robusto de errores y valores faltantes",
    "Función robusta de limpieza de precios",
    "Validación completa de integridad referencial",
    "Reportes exploratorios automáticos",
    "Manejo de datos faltantes",
    "Tipos de datos consistentes",
    "Exportación estructurada para Power BI",
    "Columna RangoEdad en tabla Clientes"
]

for mejora in mejoras:
    print(f"  ✓ {mejora}")

print("\n📁 ARCHIVOS GENERADOS:")
print(f"  • {archivo_salida} - Dataset relacional completo")

print("\n➡️  PRÓXIMOS PASOS:")
print("  1. Abrir Power BI Desktop")
print("  2. Importar el archivo Excel generado")
print("  3. Establecer relaciones entre tablas")
print("  4. Crear dashboard con métricas clave")

print("=" * 60)
print("✅ AVANCE 2 COMPLETADO EXITOSAMENTE")
print("=" * 60)

🎯 RESUMEN FINAL - AVANCE 2 COMPLETADO
MODELO RELACIONAL CREADO EXITOSAMENTE
📋 RESUMEN DE TABLAS CREADAS:
  • Ciudades        :     5 registros
  • Sucursales      :     6 registros
  • Clientes        : 17449 registros
  • Vendedores      :    30 registros
  • Productos       :    80 registros
  • Facturas        : 30000 registros
  • DetalleFacturas : 60059 registros

💰 MÉTRICAS PRINCIPALES:
  • Ventas totales: $2,572,205,600,000
  • Unidades vendidas: 495,597
  • Productos únicos vendidos: 40
  • Facturas procesadas: 30000
  • Líneas de detalle: 60059

🔄 MEJORAS INCORPORADAS:
  ✓ Lógica inteligente para asignación ciudad-sucursal
  ✓ Manejo robusto de errores y valores faltantes
  ✓ Función robusta de limpieza de precios
  ✓ Validación completa de integridad referencial
  ✓ Reportes exploratorios automáticos
  ✓ Manejo de datos faltantes
  ✓ Tipos de datos consistentes
  ✓ Exportación estructurada para Power BI
  ✓ Columna RangoEdad en tabla Clientes

📁 ARCHIVOS GENERADOS:
  • Modelo